# SWE-GraphBench: documented embeddings for the rebuilt graphs

The competition's embeddings are from an undocumented model and can only be queried with a symbol name. This notebook embeds every non-test node of the 127 rebuilt graphs with open, documented models and evaluates **natural-language issue -> symbol** retrieval on the 121 benchmark tasks, alone and fused with BM25 and PageRank.

Models (both Apache-2.0):
* `Qwen/Qwen3-Embedding-0.6B` (1024-d, instruction-aware)
* `jinaai/jina-embeddings-v2-base-code` (768-d, code-specific)

Outputs: `embeddings_<model>/<repo>_<sha>.npz` (same key->vector layout as the competition archives, float16) and `results_embeddings.json`.

Needs a GPU accelerator and internet (to download the models).

In [ ]:
import os, sys, json, time, shutil, subprocess
from pathlib import Path
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'rank_bm25', 'sentence-transformers>=3.0', 'einops'], check=True)
import numpy as np, torch
print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

COMP = Path('/kaggle/input/gemma-4-developer-agent')
def find_dir(name):
    hits = [p for p in Path('/kaggle/input').rglob(name) if p.is_dir() and 'gemma-4-developer-agent' not in str(p)]
    assert hits, name
    return sorted(hits, key=lambda p: len(str(p)))[0]
ROOT = Path('/kaggle/working/swegb'); ROOT.mkdir(exist_ok=True)
if (ROOT / 'swegraphbench').exists(): shutil.rmtree(ROOT / 'swegraphbench')
shutil.copytree(find_dir('swegraphbench'), ROOT / 'swegraphbench')
sys.path.insert(0, str(ROOT))
from swegraphbench.graphio import load_graph, graph_key
from swegraphbench.retrieval import GraphIndex, rrf, hit_at, mrr
REB = find_dir('rebuilt_graphs')
gold_path = [p for p in Path('/kaggle/input').rglob('gold.jsonl') if 'gemma-4-developer-agent' not in str(p)][0]
tasks = {json.loads(l)['instance_id']: json.loads(l) for l in open(COMP / 'tasks.jsonl', encoding='utf-8') if l.strip()}
gold = {json.loads(l)['instance_id']: json.loads(l) for l in open(gold_path, encoding='utf-8')}
RS = {'fastapi/fastapi': 'fastapi', 'Textualize/rich': 'rich', 'psf/requests': 'requests', 'encode/httpx': 'httpx'}
bench = [t for t in tasks.values() if gold[t['instance_id']]['symbols']]
print('benchmark tasks:', len(bench), '| rebuilt graphs:', len(list(REB.glob('*.json'))))

## 1. Load graphs once, collect unique node texts

In [ ]:
MAX_CHARS = 2000
def node_text(nid, text):
    return (nid + '\n' + text)[:MAX_CHARS]

indexes = {}   # key -> GraphIndex (tests filtered)
unique = {}    # text -> row id
node_rows = {} # key -> np.array of row ids aligned with idx.nodes
t0 = time.time()
for t in bench:
    key = (RS[t['repo']], t['base_commit'])
    if key in indexes: continue
    g = load_graph(REB / f'{key[0]}_{key[1]}.json')
    idx = GraphIndex(g, exclude_tests=True)
    rows = []
    for n in idx.nodes:
        s = node_text(n, idx.text[n])
        rows.append(unique.setdefault(s, len(unique)))
    indexes[key] = idx; node_rows[key] = np.array(rows)
texts = list(unique.keys())
print(f'{len(indexes)} graphs, {sum(len(i.nodes) for i in indexes.values())} node slots, {len(texts)} unique texts, {time.time()-t0:.0f}s')

## 2. Embed with each model, evaluate dense and fused retrieval

In [ ]:
from sentence_transformers import SentenceTransformer
KS = (1, 5, 10, 20)
MODELS = {
    'qwen3-0.6b': dict(name='Qwen/Qwen3-Embedding-0.6B', query_prompt='Instruct: Given a GitHub issue, retrieve the function or class that must be edited to fix it\nQuery: ', doc_prompt='', kw=dict(model_kwargs={'torch_dtype': torch.float16})),
    'jina-v2-code': dict(name='jinaai/jina-embeddings-v2-base-code', query_prompt='', doc_prompt='', kw=dict(trust_remote_code=True, model_kwargs={'torch_dtype': torch.float16})),
}

def evaluate(ranked, gall, ids):
    gin = {s for s in gall if s in ids}
    return {f'recall@{k}': len(set(ranked[:k]) & gin) / len(gall) for k in KS} | {f'hit@{k}': hit_at(ranked, gin, k) for k in KS} | {'mrr': mrr(ranked, gin)}

results = {}
for tag, cfg in MODELS.items():
    print('=== ', tag, cfg['name'], flush=True)
    model = SentenceTransformer(cfg['name'], device='cuda', **cfg['kw'])
    model.max_seq_length = 512
    t0 = time.time()
    D = model.encode([cfg['doc_prompt'] + s for s in texts], batch_size=64, normalize_embeddings=True, show_progress_bar=True, convert_to_numpy=True).astype(np.float16)
    print(f'docs embedded: {D.shape} in {time.time()-t0:.0f}s', flush=True)
    Q = model.encode([cfg['query_prompt'] + t['problem_statement'][:4000] for t in bench], batch_size=16, normalize_embeddings=True, convert_to_numpy=True)
    # save per-graph archives in the competition layout
    out_dir = Path(f'/kaggle/working/embeddings_{tag}'); out_dir.mkdir(exist_ok=True)
    for key, idx in indexes.items():
        np.savez_compressed(out_dir / f'{key[0]}_{key[1]}.npz', **{n: D[r] for n, r in zip(idx.nodes, node_rows[key])})
    rows = []
    for qi, t in enumerate(bench):
        key = (RS[t['repo']], t['base_commit']); idx = indexes[key]; ids = set(idx.nodes)
        gall = set(gold[t['instance_id']]['symbols'])
        sims = D[node_rows[key]].astype(np.float32) @ Q[qi]
        order = np.argsort(-sims)[:max(KS)]
        dense = [idx.nodes[i] for i in order]
        bm = idx.bm25_text(t['problem_statement'], max(KS)); pp = idx.ppr(t['problem_statement'], max(KS))
        r = {'instance_id': t['instance_id'], 'repo': key[0], 'methods': {
            'bm25_text': evaluate(bm, gall, ids), 'ppr': evaluate(pp, gall, ids),
            'hybrid_lexical': evaluate(rrf([bm, pp], max(KS)), gall, ids),
            'dense': evaluate(dense, gall, ids),
            'hybrid_dense': evaluate(rrf([bm, pp, dense], max(KS)), gall, ids),
            'bm25+dense': evaluate(rrf([bm, dense], max(KS)), gall, ids)}}
        rows.append(r)
    results[tag] = {'model': cfg['name'], 'dim': int(D.shape[1]), 'n_unique_texts': len(texts), 'rows': rows}
    del model; torch.cuda.empty_cache()
json.dump(results, open('/kaggle/working/results_embeddings.json', 'w'))
print('saved')

## 3. Summary tables (all 121 tasks, and the 65 with a released graph)

In [ ]:
import random
def boot(vals, n=2000, seed=0):
    rng = random.Random(seed); m = []
    for _ in range(n):
        s = [vals[rng.randrange(len(vals))] for _ in vals]; m.append(sum(s)/len(s))
    m.sort(); return m[int(.025*n)], m[int(.975*n)]
released = {t['instance_id'] for t in bench if (COMP / 'graphs' / f"{RS[t['repo']]}_{t['base_commit']}.json").exists()}
summary = {}
for tag, res in results.items():
    for subset, rows in [('all121', res['rows']), ('released65', [r for r in res['rows'] if r['instance_id'] in released]), ('norel56', [r for r in res['rows'] if r['instance_id'] not in released])]:
        print(f'\n[{tag}] {subset} n={len(rows)}')
        print(f"{'method':16s} {'H@1':>6s} {'H@5':>6s} {'R@10':>6s} {'R@20':>6s} {'MRR':>6s}   95% CI of H@5")
        for m in ['bm25_text', 'ppr', 'hybrid_lexical', 'dense', 'bm25+dense', 'hybrid_dense']:
            vals = {k: [r['methods'][m][k] for r in rows] for k in ['hit@1', 'hit@5', 'recall@10', 'recall@20', 'mrr']}
            means = {k: sum(v)/len(v) for k, v in vals.items()}; lo, hi = boot(vals['hit@5'])
            summary[f'{tag}/{subset}/{m}'] = means | {'hit@5_ci95': [lo, hi], 'n': len(rows)}
            print(f"{m:16s} {means['hit@1']:6.3f} {means['hit@5']:6.3f} {means['recall@10']:6.3f} {means['recall@20']:6.3f} {means['mrr']:6.3f}   [{lo:.3f}, {hi:.3f}]")
json.dump(summary, open('/kaggle/working/summary_embeddings.json', 'w'), indent=1)
print('\nunique texts embedded:', {k: v['n_unique_texts'] for k, v in results.items()})